# 01. Khám Phá Dữ Liệu (EDA), Tiền Xử Lý & Chọn Lọc Đặc Trưng (Phương Án A)
**Đề tài:** Hệ thống Cảnh báo sớm Nguy cơ mắc bệnh Tiểu đường dựa trên Chỉ số Sức khỏe và Thói quen Sinh hoạt (CO3117 - Học Máy)  
**Phụ trách:** Người 1 (Data & EDA)

---
### Quy trình thực hiện trong Notebook này:
1. **Khảo sát chi tiết 3 tệp dữ liệu thô (`data/raw/`)** và kiểm chứng sự khác biệt ánh xạ nhãn của nhóm Tiền tiểu đường (`Prediabetes`).
2. **Định nghĩa nhãn mục tiêu theo Phương án A (Chuẩn Cảnh báo sớm):** Gộp lớp `1.0` (*Tiền tiểu đường*: $4,631$ mẫu) và lớp `2.0` (*Tiểu đường*: $35,346$ mẫu) từ `diabetes_012_health_indicators_BRFSS2015.csv` thành **Nhãn `1` (Nguy cơ / Mắc bệnh: $39,977$ mẫu - $15.76\%$)**, giữ lớp `0.0` làm **Nhãn `0` (Khỏe mạnh: $213,703$ mẫu - $84.24\%$)**.
3. **Làm sạch dữ liệu (Data Cleaning):**
   - Loại bỏ $599$ bản ghi dị thường $\text{BMI} \ge 70$ (xem figure 3, bmi > 70 nhưng tỉ lệ béo phì lại giảm chỉ còn xấp xỉ 13% và bmi >70 được xem là rất cao - không hợp lý đối với bình thường - nên có thể đây là nhiễu hoặc lỗi nhập liệu).
   - Loại bỏ bản ghi trùng lặp hoàn toàn (`drop_duplicates` — $23,968$ mẫu) để ngăn ngừa rò rỉ dữ liệu (*Data Leakage*).
   - Giải quyết xung đột nhãn biên quyết định (*Label Conflicts*: các tổ hợp 21 đặc trưng giống hệt nhau nhưng vừa mang nhãn `0` vừa mang nhãn `1` — loại bỏ $1,804$ mẫu nhiễu âm tính), ưu tiên giữ nhãn `1` để tối ưu độ nhạy cảnh báo sớm - hướng mô hình cảnh báo bệnh bao quát hơn, hành vi y tế tiêu biểu.
4. **Kiểm định Đa cộng tuyến (VIF) & Chọn lọc Đặc trưng Đa tiêu chí (Feature Selection):** Kết hợp *Pearson $|r|$*, *Mutual Information*, và *Random Forest Gini Importance* để chọn ra **Top 13 đặc trưng cốt lõi** cho bộ `Selected`.
5. **Kiểm tra chất lượng dữ liệu đầu ra (Quality Audit) & Xuất đúng 2 bộ dữ liệu sạch** (`diabetes_full.csv` và `diabetes_selected.csv`) vào `data/processed/`.
6. **Chiến lược Xử lý Mất cân bằng Lớp (Class Imbalance):** Hướng dẫn áp dụng `class_weight` / `Resampling` an toàn (không gây Data Leakage) khi huấn luyện mô hình.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import inv
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

RAW_DIR = '../data/raw'
PROC_DIR = '../data/processed'
FIG_DIR = './figures'
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Khảo Sát 3 Tệp Dữ Liệu Thô & Định Nghĩa Nhãn Mục Tiêu (Phương Án A)

In [2]:
df_012 = pd.read_csv(os.path.join(RAW_DIR, 'diabetes_012_health_indicators_BRFSS2015.csv'))
df_bin_raw = pd.read_csv(os.path.join(RAW_DIR, 'diabetes_binary_health_indicators_BRFSS2015.csv'))
df_5050 = pd.read_csv(os.path.join(RAW_DIR, 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'))

print('1. diabetes_012 shape           :', df_012.shape)
print('2. diabetes_binary shape        :', df_bin_raw.shape)
print('3. diabetes_binary_5050split    :', df_5050.shape)
print('\nBảng đối chiếu chéo giữa Diabetes_012 và Diabetes_binary gốc trên Kaggle:')
print(pd.crosstab(df_012['Diabetes_012'], df_bin_raw['Diabetes_binary']))

1. diabetes_012 shape           : (253680, 22)
2. diabetes_binary shape        : (253680, 22)
3. diabetes_binary_5050split    : (70692, 22)

Bảng đối chiếu chéo giữa Diabetes_012 và Diabetes_binary gốc trên Kaggle:
Diabetes_binary     0.0    1.0
Diabetes_012                  
0.0              213703      0
1.0                4631      0
2.0                   0  35346


In [3]:
# Áp dụng Phương án A: Gộp Tiền tiểu đường (1.0) và Tiểu đường (2.0) thành nhãn 1 (Nguy cơ / Mắc bệnh)
orig_features = [c for c in df_012.columns if c != 'Diabetes_012']
df = df_012.copy()
df['Diabetes_binary'] = (df['Diabetes_012'] >= 1.0).astype(np.int8)
df = df.drop(columns=['Diabetes_012'])
df = df[['Diabetes_binary'] + orig_features]

# Ép kiểu nguyên để tối ưu bộ nhớ
for col in orig_features:
    df[col] = df[col].astype(np.int16 if col == 'BMI' else np.int8)

print('Phân phối nhãn theo Phương án A (Trước khi làm sạch):')
vc = df['Diabetes_binary'].value_counts()
vp = (df['Diabetes_binary'].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({'Số mẫu': vc, 'Tỷ lệ (%)': vp}))

Phân phối nhãn theo Phương án A (Trước khi làm sạch):
                 Số mẫu  Tỷ lệ (%)
Diabetes_binary                   
0                213703      84.24
1                 39977      15.76


## 2. Làm Sạch Dữ Liệu: Xử Lý Ngoại Lai BMI, Bản Ghi Trùng Lặp & Xung Đột Nhãn

In [4]:
# Bước 2.1: Lọc bỏ 599 bản ghi có BMI >= 70 (Ngoại lai do lỗi quy đổi đơn vị / nhập liệu khảo sát)
df_bmi_clean = df[df['BMI'] < 70].copy()
print(f'1. Sau khi lọc ngoại lai BMI >= 70: còn {len(df_bmi_clean):,} mẫu (đã loại {len(df) - len(df_bmi_clean):,} mẫu)')

# Bước 2.2: Khử các bản ghi trùng lặp hoàn toàn (Exact Duplicates)
df_dedup = df_bmi_clean.drop_duplicates(keep='first').copy()
print(f'2. Sau khi khử trùng lặp hoàn toàn (Exact Dedup): còn {len(df_dedup):,} mẫu (đã loại {len(df_bmi_clean) - len(df_dedup):,} mẫu)')

# Bước 2.3: Xử lý xung đột nhãn (Cùng 21 đặc trưng nhưng vừa có nhãn 0 vừa có nhãn 1)
# Ưu tiên giữ nhãn 1 (Nguy cơ) để làm sạch biên quyết định và tăng độ nhạy (Recall) cho Hệ thống Cảnh báo sớm
df_clean = (
    df_dedup.sort_values('Diabetes_binary', ascending=False)
    .drop_duplicates(subset=orig_features, keep='first')
    .sort_index()
    .reset_index(drop=True)
)
print(f'3. Sau khi giải quyết xung đột nhãn biên quyết định: còn {len(df_clean):,} mẫu (đã loại {len(df_dedup) - len(df_clean):,} mẫu nhiễu âm tính)')
print('\nPhân phối nhãn sạch cuối cùng (Clean Final Dataset):')
print(pd.DataFrame({
    'Số mẫu': df_clean['Diabetes_binary'].value_counts(),
    'Tỷ lệ (%)': (df_clean['Diabetes_binary'].value_counts(normalize=True)*100).round(2)
}))

1. Sau khi lọc ngoại lai BMI >= 70: còn 253,081 mẫu (đã loại 599 mẫu)
2. Sau khi khử trùng lặp hoàn toàn (Exact Dedup): còn 229,113 mẫu (đã loại 23,968 mẫu)
3. Sau khi giải quyết xung đột nhãn biên quyết định: còn 227,309 mẫu (đã loại 1,804 mẫu nhiễu âm tính)

Phân phối nhãn sạch cuối cùng (Clean Final Dataset):
                 Số mẫu  Tỷ lệ (%)
Diabetes_binary                   
0                187745      82.59
1                 39564      17.41


## 3. Kiểm Định Đa Cộng Tuyến (VIF) & Chọn Lọc Đặc Trưng Đa Tiêu Chí

In [ ]:
X_clean = df_clean[orig_features]
y_clean = df_clean['Diabetes_binary']

# Tính VIF
X_std = (X_clean - X_clean.mean()) / X_clean.std()
vif_vals = np.diag(inv(X_std.corr().values))

# Tính Pearson |r|, ANOVA F-stat, Mutual Information và Random Forest Importance

pearson_r = X_clean.apply(lambda col: col.corr(y_clean))
pearson_abs = pearson_r.abs()
f_vals, _ = f_classif(X_clean, y_clean)

samp = df_clean.sample(n=40000, random_state=42)
mi_vals = mutual_info_classif(samp[orig_features], samp['Diabetes_binary'], discrete_features=True, random_state=42)

rf = RandomForestClassifier(n_estimators=80, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_clean, y_clean)
rf_vals = rf.feature_importances_

ranking_df = pd.DataFrame({
    'VIF': vif_vals,
    'Pearson_r': pearson_r,
    'F_Stat': f_vals,
    'Pearson_Score': pearson_abs / pearson_abs.max() * 100,
    'MI_Score': mi_vals / mi_vals.max() * 100,
    'RF_Score': rf_vals / rf_vals.max() * 100
}, index=orig_features)
ranking_df['Composite_Score'] = ranking_df[['Pearson_Score', 'MI_Score', 'RF_Score']].mean(axis=1)
ranking_df = ranking_df.sort_values('Composite_Score', ascending=False)

selected_features = [
    'GenHlth', 'HighBP', 'BMI', 'HighChol', 'Age', 'DiffWalk', 'PhysHlth',
    'HeartDiseaseorAttack', 'Income', 'Education', 'PhysActivity', 'Stroke', 'MentHlth'
]
dropped_features = [c for c in orig_features if c not in selected_features]
ranking_df['Decision'] = ['KEEP (Selected)' if f in selected_features else 'DROP (Low Signal)' for f in ranking_df.index]
print(ranking_df.round(3).to_string())

                        VIF  Pearson_r     F_Stat  Pearson_Score  MI_Score  RF_Score  Composite_Score           Decision
GenHlth               1.722      0.282  19643.233        100.000   100.000   100.000          100.000    KEEP (Selected)
HighBP                1.308      0.262  16804.055         93.028    87.397    92.354           90.926    KEEP (Selected)
BMI                   1.142      0.231  12976.599         81.940    72.078    93.176           82.398    KEEP (Selected)
HighChol              1.158      0.206  10101.359         73.138    55.953    53.304           60.798    KEEP (Selected)
Age                   1.355      0.187   8263.357         66.407    60.381    50.894           59.227    KEEP (Selected)
DiffWalk              1.512      0.207  10179.348         73.407    46.642    30.112           50.054    KEEP (Selected)
PhysHlth              1.593      0.157   5747.674         55.682    29.278    23.071           36.011    KEEP (Selected)
HeartDiseaseorAttack  1.168     

In [12]:
print(ranking_df)

                           VIF  Pearson_r        F_Stat  Pearson_Score  \
GenHlth               1.721564   0.282034  19643.232775     100.000000   
HighBP                1.308321   0.262369  16804.055070      93.027573   
BMI                   1.142309   0.231099  12976.598647      81.940110   
HighChol              1.158466   0.206273  10101.358743      73.137512   
Age                   1.355219   0.187291   8263.356862      66.407198   
DiffWalk              1.512293   0.207034  10179.348252      73.407258   
PhysHlth              1.592718   0.157043   5747.673746      55.682113   
HeartDiseaseorAttack  1.167651   0.166926   6515.261852      59.186318   
Income                1.426156  -0.146221   4966.086327      51.844981   
Education             1.270212  -0.107442   2654.841794      38.095283   
PhysActivity          1.129707  -0.101000   2342.664908      35.811241   
Stroke                1.077646   0.097161   2166.271039      34.449958   
CholCheck             1.035248   0.077

## 4. Kiểm Tra Chất Lượng Dữ Liệu (Data Quality Audit) & Xuất 2 Bộ Data (`Full` & `Selected`)

In [13]:
df_full = df_clean[['Diabetes_binary'] + orig_features].copy()
df_selected = df_clean[['Diabetes_binary'] + selected_features].copy()

# Kiểm định tự động (Assertions)
assert df_full.isnull().sum().sum() == 0, 'Còn giá trị NaN!'
assert df_full.duplicated().sum() == 0, 'Còn bản ghi trùng lặp!'
assert df_full.duplicated(subset=orig_features).sum() == 0, 'Còn xung đột nhãn trên 21 biến!'
assert df_full['BMI'].between(12, 69).all(), 'BMI vượt ngoài ngưỡng sạch [12, 69]!'
assert (df_full['Diabetes_binary'] == df_selected['Diabetes_binary']).all(), 'Hai bộ Full và Selected không khớp thứ tự dòng!'

full_path = os.path.join(PROC_DIR, 'diabetes_full.csv')
sel_path = os.path.join(PROC_DIR, 'diabetes_selected.csv')
df_full.to_csv(full_path, index=False)
df_selected.to_csv(sel_path, index=False)

print('KIỂM ĐỊNH ĐẠT 100%! Đã xuất 2 bộ dữ liệu sạch vào data/processed/:')
print(f'  1. diabetes_full.csv     : shape={df_full.shape}     | {os.path.getsize(full_path)/1024/1024:.2f} MB')
print(f'  2. diabetes_selected.csv : shape={df_selected.shape}     | {os.path.getsize(sel_path)/1024/1024:.2f} MB')

KIỂM ĐỊNH ĐẠT 100%! Đã xuất 2 bộ dữ liệu sạch vào data/processed/:
  1. diabetes_full.csv     : shape=(227309, 22)     | 9.90 MB
  2. diabetes_selected.csv : shape=(227309, 14)     | 6.43 MB


## 5. Lưu Ý Kỹ Thuật Xử Lý Mất Cân Bằng Lớp (82.59% Lớp 0 vs 17.41% Lớp 1)
- **Tại sao không ghi đè SMOTE/Undersampling trực tiếp vào 2 file CSV trên?**  
  Vì nếu Resample toàn bộ dataset trước khi tách Train/Test, các mẫu nhân tạo sẽ gây **Data Leakage** sang tập Test và làm sai lệch phân phối thực tế khi đánh giá PR-AUC, Recall, $F_2\text{-Score}$.
- **Quy chuẩn thống nhất cho Người 2, 3, 4 khi đọc `diabetes_full.csv` và `diabetes_selected.csv`:**
  1. Dùng chung `train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)` để hai bộ `Full` và `Selected` (cũng như mọi mô hình A, B, MLP, Ensemble) khớp tuyệt đối tập Test.
  2. Áp dụng trọng số phạt chi phí (`class_weight='balanced'` hoặc `scale_pos_weight ≈ 4.75`) **HOẶC** dùng `imblearn.pipeline.Pipeline` (ví dụ `RandomUnderSampler` / `TomekLinks` / `SMOTE`) **chỉ trên tập Train**.